In [633]:
!pip install gensim

In [634]:
import kagglehub
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.


In [698]:
import os
os.listdir(path)

['spam.csv']

In [699]:
import pandas as pd
df=pd.read_csv(path+'/spam.csv',encoding='latin-1',usecols=['v1','v2'])

In [700]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [701]:
df.shape

(5572, 2)

In [702]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   v1      5572 non-null   object
 1   v2      5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [703]:
df.isnull().sum()

,0
v1,0
v2,0


In [704]:
df.duplicated().sum()

np.int64(403)

In [705]:
df.drop_duplicates(inplace=True)

In [706]:
df.sample()

,v1,v2
3356,ham,"Brainless Baby Doll..:-D;-), vehicle sariyag d..."


In [707]:
df['v2'][0]

'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'

In [708]:
df.shape

(5169, 2)

In [709]:
import re
def remove_unwanted(col):
  col=col.lower()
  col=re.sub('<.*?>',' ',col)
  col=re.sub('http?://\S+|www\.\S+',' ',col)
  return re.sub('[^a-zA-Z]',' ',col)
df['v2']=df['v2'].apply(remove_unwanted)

<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:5: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_2385/1710675825.py:5: SyntaxWarning: invalid escape sequence '\S'
  col=re.sub('http?://\S+|www\.\S+',' ',col)


In [710]:
df['v2'][0]

'go until jurong point  crazy   available only in bugis n great world la e buffet    cine there got amore wat   '

In [711]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize,sent_tokenize
def remove_stopwords(col):
  L=[]
  words=stopwords.words('english')
  negations=['not','never','isnt','wouldnt','wasnt','havent','arent']
  tokens=word_tokenize(col)
  for token in tokens:
    if token not in words or token in negations:
      L.append(token)
  return " ".join(L)
df['v2']=df['v2'].apply(remove_stopwords)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [712]:
df['v2'][0]

'go jurong point crazy available bugis n great world la e buffet cine got amore wat'

In [713]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
def lemmatize(col):
  lemmatizer=WordNetLemmatizer()
  tokens=word_tokenize(col)
  L=[lemmatizer.lemmatize(token) for token in tokens]
  return L
df['v2']=df['v2'].apply(lemmatize)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [714]:
print(df['v2'][0])

['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']


In [715]:
df.shape

(5169, 2)

In [716]:
corpus=list(df['v2'])

In [717]:
len(corpus)

5169

In [718]:
from gensim.models import Word2Vec
model=Word2Vec(window=5,min_count=2)
model.build_vocab(corpus)

In [719]:
model.train(corpus,total_examples=model.corpus_count,epochs=30)

(1130013, 1368510)

In [720]:
len(model.wv.index_to_key)

3259

In [721]:
print(model.wv.index_to_key)

['u', 'call', 'not', 'get', 'ur', 'go', 'gt', 'lt', 'ok', 'free', 'know', 'day', 'got', 'time', 'like', 'come', 'good', 'want', 'text', 'love', 'need', 'one', 'send', 'p', 'going', 'r', 'today', 'lor', 'home', 'txt', 'stop', 'see', 'c', 'still', 'n', 'k', 'back', 'take', 'da', 'think', 'tell', 'reply', 'mobile', 'dont', 'hi', 'new', 'sorry', 'phone', 'please', 'week', 'make', 'night', 'min', 'msg', 'well', 'thing', 'much', 'hey', 'hope', 'oh', 'say', 'na', 'great', 'give', 'wat', 'number', 'dear', 'later', 'pls', 'b', 'message', 'claim', 'way', 'happy', 'co', 'wan', 'work', 'let', 'yes', 'friend', 'already', 'e', 'ask', 'yeah', 'babe', 'really', 'prize', 'im', 'right', 'tomorrow', 'meet', 'find', 'thanks', 'cash', 'said', 'life', 'would', 'lol', 'tone', 'last', 'anything', 'miss', 'also', 'year', 'sure', 'service', 'something', 'morning', 'keep', 'every', 'care', 'feel', 'contact', 'wait', 'urgent', 'buy', 'next', 'sent', 'nokia', 'x', 'first', 'around', 'money', 'pick', 'tonight', 'wi

In [722]:
model.wv.most_similar('good')

[('morning', 0.9166931509971619),
 ('sweet', 0.8836358189582825),
 ('brings', 0.8822994232177734),
 ('wonderful', 0.877464771270752),
 ('dream', 0.8672756552696228),
 ('evening', 0.8671788573265076),
 ('boy', 0.8670039772987366),
 ('wish', 0.8642450571060181),
 ('thank', 0.8627658486366272),
 ('moment', 0.8598708510398865)]

In [723]:
model.wv.similarity('good','bad')

np.float32(0.72467655)

In [724]:
model.wv['good']

array([-0.12808362,  0.98341614,  0.01692561,  0.49025425,  0.7631981 ,
       -0.67852765, -0.23106943,  2.1340342 , -1.0086182 , -0.8303638 ,
        0.01874692,  0.3145375 , -0.2387289 ,  0.32480386, -0.3223303 ,
        0.09282782, -0.53775483, -0.66933745, -0.00643021, -0.84965885,
       -0.36345345,  0.07141218,  0.86184484,  0.23143366, -0.2864976 ,
       -0.3056148 ,  0.3568045 , -1.2087166 , -0.3228832 ,  0.01909257,
       -0.04013044, -0.48323295,  0.31544098, -0.77004176, -0.26742992,
        0.04172108, -0.12357591, -0.8570926 , -0.2135204 , -0.94383115,
        1.0229602 , -0.26411366, -0.7162658 ,  0.608689  ,  1.0385032 ,
        0.04571111, -0.33114368, -1.161867  ,  0.524595  ,  0.5425257 ,
        0.49737495, -0.5170963 , -0.31915465, -0.14786847, -0.6523805 ,
       -0.36610293, -0.8136316 , -0.49479318,  0.05359642, -1.0642127 ,
        0.647806  ,  0.7809969 , -0.29764402,  0.3161434 , -0.4596751 ,
        0.26596725,  0.3540515 , -0.10124865, -0.46070024,  0.51

In [730]:
import joblib
with open('preprocessor.pkl','wb') as f:
  joblib.dump(model,f)

In [731]:
print(corpus[0])

['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']


In [732]:
df.reset_index(inplace=True,drop=True)

In [733]:
df.index

RangeIndex(start=0, stop=5169, step=1)

In [735]:
import numpy as np
def AvgW2V(col):
  return np.mean([model.wv[token] for token in col if token in model.wv.index_to_key],axis=0)

In [736]:
len(AvgW2V(corpus[0]))

100

In [737]:
X=[]
for i in range(len(corpus)):
  X.append(AvgW2V(corpus[i]))

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [738]:
len(X)

5169

In [739]:
X[0].shape

(100,)

In [740]:
type(X[0])

numpy.ndarray

In [741]:
nan_values=[]
for i,j in enumerate(X):
  if type(j)!=type(X[0]):
    nan_values.append(i)
    print(i,j,end=" ")

248 nan 442 nan 500 nan 766 nan 873 nan 939 nan 1400 nan 1540 nan 1560 nan 1773 nan 1920 nan 2458 nan 3191 nan 3522 nan 4012 nan 4188 nan 4500 nan 

In [742]:
df.reset_index(inplace=True,drop=True)

In [743]:
a=0
for i in nan_values:
  X.pop(i-a)
  df.drop(i,inplace=True)
  a+=1

In [744]:
df.reset_index(inplace=True,drop=True)

In [745]:
# Dependent Feature
from sklearn.preprocessing import LabelEncoder
label=LabelEncoder()
y=label.fit_transform(df['v1'])

In [746]:
y

array([0, 0, 1, ..., 0, 0, 0], shape=(5152,))

In [747]:
## 1 -> Spam
## 0 -> Ham

In [748]:
X[0]

array([-0.4374803 ,  0.36987045, -0.05707126, -0.2504604 ,  0.11408424,
       -0.6524523 , -0.02334883,  0.72258264,  0.08544499, -0.06359531,
       -0.33065438, -0.55503786, -0.3850229 , -0.00576134,  0.07112207,
       -0.47856325,  0.01290592, -0.39466903,  0.13755383, -0.32225865,
        0.2948563 ,  0.03486141,  0.10913333,  0.24731736,  0.09572104,
       -0.06169686, -0.12413752, -0.2479357 , -0.15124062,  0.09277381,
        0.24208999, -0.18279633,  0.16152313, -0.03795362, -0.08018369,
        0.3417459 ,  0.11120204, -0.2396746 ,  0.10299945, -0.4917364 ,
       -0.41475582, -0.17010842,  0.03292779, -0.15069246,  0.09148065,
       -0.13807864, -0.09340926,  0.30810255,  0.03694414,  0.30111313,
        0.27881444, -0.2599462 ,  0.22313848, -0.03645018,  0.02737259,
        0.03723736,  0.6713404 ,  0.16101691, -0.3846178 ,  0.23486136,
       -0.02434689,  0.02025394,  0.00182476, -0.32767296, -0.505656  ,
        0.3290259 ,  0.28184804,  0.34227178, -0.4725677 ,  0.61

In [749]:
df=pd.DataFrame(X)

In [750]:
df.shape

(5152, 100)

In [751]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df,y,test_size=0.2,random_state=42)

In [752]:
from sklearn.naive_bayes import BernoulliNB
model=BernoulliNB()
model.fit(x_train,y_train)

BernoulliNB()

In [753]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
train_pred=model.predict(x_train)
test_pred=model.predict(x_test)
print("Training Accuracy : ",accuracy_score(y_train,train_pred))
print("Testing Accuracy : ",accuracy_score(y_test,test_pred))

Training Accuracy :  0.9403057510313031
Testing Accuracy :  0.9379243452958292


In [754]:
print("For Training : \n",confusion_matrix(y_train,train_pred))

For Training : 
 [[3370  221]
 [  25  505]]


In [755]:
print("For Testing : \n",confusion_matrix(y_test,test_pred))

For Testing : 
 [[859  49]
 [ 15 108]]


In [756]:
print('Train Report :',classification_report(y_train,train_pred))
print('Test Report :',classification_report(y_test,test_pred))

Train Report :               precision    recall  f1-score   support

           0       0.99      0.94      0.96      3591
           1       0.70      0.95      0.80       530

    accuracy                           0.94      4121
   macro avg       0.84      0.95      0.88      4121
weighted avg       0.95      0.94      0.94      4121

Test Report :               precision    recall  f1-score   support

           0       0.98      0.95      0.96       908
           1       0.69      0.88      0.77       123

    accuracy                           0.94      1031
   macro avg       0.84      0.91      0.87      1031
weighted avg       0.95      0.94      0.94      1031



In [757]:
with open('model.pkl','wb') as f:
  joblib.dump(model,f)

In [697]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 18.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 33.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]
